# ResNet 18

In [2]:
import torch.nn as nn  # 모델 레이어
import torch.nn.functional as F  # softmax 등 함수
import torchvision  # vision dataset/util
import torchvision.transforms as T  # transform
import torchvision.models as models  # resnet18
from torch.utils.data import DataLoader  # 데이터 로더

In [ ]:
## CIFAR-10에서 널리 쓰는 normalize 값
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)  # 채널별 평균(R,G,B)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)  # 채널별 표준편차(R,G,B)

train_tfms = T.Compose([  # train transform(augmentation 포함)
    T.???(32, padding=4),  # RandomCrop 32x32를 padding 후 random crop
    T.???(),  # RandomHorizontalFlip 좌우 반전
    T.???(),  # ToTensor PIL -> torch tensor (C,H,W), 0~1
    T.???(CIFAR10_MEAN, CIFAR10_STD),  # Normalize 정규화, -2.5~2.5
])

test_tfms = T.Compose([  # test transform(augmentation 없음)
    T.ToTensor(),  # 텐서 변환
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),  # 정규화
])
data_root = './data'  # 데이터 저장 경로
train_set = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tfms)  # train set
test_set  = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tfms)  # test set

class_names = train_set.classes  # 클래스 이름 목록
num_classes = len(class_names)  # 클래스 개수(10)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)  # train loader
test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)  # test loader

def denorm(x): # -2.5~2.5
    mean = torch.tensor(CIFAR10_MEAN).view(1,3,1,1)  # (1,3,1,1) 형태로 broadcast
    std  = torch.tensor(CIFAR10_STD).view(1,3,1,1)  # (1,3,1,1) 형태로 broadcast
    return x * std + mean  # 정규화 역변환, 0~1

images, labels = next(iter(train_loader))  # 한 배치 가져오기
images_dn = denorm(images[:16]).clamp(0,1)  # 앞 16장만 역정규화 후 [0,1]로 clip

## ImageNet용 stem(7×7 stride 2 + maxpool)은 conv1을 3×3 stride 1로 바꾸고 maxpool을 제거합니다
def build_resnet18_for_cifar10(num_classes: int = 10):
    model = models.resnet18(weights=None)  # ImageNet pretrained 없이 ResNet18 생성 (from scratch)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)  # CIFAR용 stem
    model.maxpool = nn.Identity()  # maxpool 제거
    model.fc = nn.Linear(model.fc.in_features, num_classes)  # 최종 분류기 출력=10
    return model

def accuracy_top1(logits, targets):
    preds = logits.argmax(dim=1)  # 가장 큰 logit의 클래스 선택, logits = [B, C], preds = [B]
    return (preds == targets).float().mean().item()  # top-1 accuracy, scalar

scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())  # AMP용 scaler, 학습 속도를 높이고 메모리 사용량을 줄임
print(scaler)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()  # train mode(dropout/bn 동작 변경)
    total_loss, total_acc = 0.0, 0.0  # 누적 loss/acc
    n = 0  # 샘플 수 누적
    t0 = time.time()  # 시간 측정 시작
    for images, labels in loader:
        images = images.to(device, non_blocking=True)  # 입력을 GPU로 이동 (Asynchronous transfer)
        labels = labels.to(device, non_blocking=True)  # 라벨을 GPU로 이동 (Asynchronous transfer)

        optimizer.zero_grad(set_to_none=True)  # gradient 초기화
        with torch.amp.autocast(device_type=device.type, enabled=torch.cuda.is_available()):  # AMP autocast
            logits = model(images)  # forward
            loss = criterion(logits, labels)  # loss 계산
            
        scaler.scale(loss).backward()  # scaled backward
        scaler.step(optimizer)  # optimizer step
        scaler.update()  # scaler 업데이트

        bs = images.size(0)  # batch size
        total_loss += loss.item() * bs  # batch loss 누적
        total_acc  += accuracy_top1(logits.detach(), labels) * bs  # logits값만 복사, batch acc 누적
        n += bs  # 샘플 수 누적

    dt = time.time() - t0  # epoch 수행 시간
    return total_loss / n, total_acc / n, dt  # 평균 loss/acc/시간

@torch.no_grad()   # Autograd off, 함수 내부에서 기울기 계산 멈춤
def evaluate(model, loader, criterion):
    model.eval()  # eval mode, i.e., Dropout 비활성화, BN Running Stats 고정
    total_loss, total_acc = 0.0, 0.0  # 누적
    n = 0  # 샘플 수
    for images, labels in loader:
        images = images.to(device, non_blocking=True)  # GPU 이동
        labels = labels.to(device, non_blocking=True)  # GPU 이동
        logits = model(images)  # forward
        loss = criterion(logits, labels)  # loss
        bs = images.size(0)  # batch size
        total_loss += loss.item() * bs  # 누적
        total_acc  += accuracy_top1(logits, labels) * bs  # 누적
        n += bs  # 누적
    return total_loss / n, total_acc / n  # 평균 loss/acc

@torch.no_grad()  # 자동 미분(Autograd) 기능 비활성화
def predict_batch(model, images):
    logits = model(images)  # raw logits
    probs = F.softmax(logits, dim=1)  # 확률로 변환
    conf, pred = probs.max(dim=1)  # 최대 확률(conf)과 클래스(pred)
    return pred, conf, probs

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)  # 분류 loss
optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr, momentum=cfg.momentum, weight_decay=cfg.weight_decay)  # Mini-Batch SGD optimizer with Momentum

# 학습률을 T_max에 맞춰 코사인 곡선 형태로 변화
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs_to_run)  # 이번 실험 구간에 대해 cosine schedule

for e in range(1, epochs_to_run+1):
    epoch = start_epoch + e  # 실제 epoch 번호(누적)
    lr_now = optimizer.param_groups[0]['lr']  # 현재 lr, [ { 'params': [...], 'lr': 0.1, 'momentum': 0.9, 'weight_decay': 0.0005, ... } ]
    tr_loss, tr_acc, dt = train_one_epoch(model, train_loader, optimizer, criterion)  # 1 epoch train
    va_loss, va_acc = evaluate(model, test_loader, criterion)  # validation
    scheduler.step()  # lr 스케줄 업데이트

images, labels = next(iter(test_loader))  # test 배치 하나
images = images.to(device)  # GPU 이동
labels = labels.to(device)  # GPU 이동
pred, conf, probs = predict_batch(model, images)  # 예측

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()  # eval 모드
    all_preds, all_labels, all_images = [], [], []  # 누적 리스트
    for images, labels in loader:
        images = images.to(device)  # GPU 이동
        labels = labels.to(device)  # GPU 이동
        preds, conf, _ = predict_batch(model, images)  # 예측
        all_preds.append(preds.cpu())  # CPU로 모아두기
        all_labels.append(labels.cpu())  # CPU로 모아두기
        all_images.append(images.cpu())  # 이미지도 저장(오분류 시각화용)
    return torch.cat(all_images), torch.cat(all_preds), torch.cat(all_labels)  # 전체 텐서로 결합

def confusion_and_perclass(preds, labels, num_classes):
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)  # confusion matrix 초기화
    for t, p in zip(labels, preds):
        cm[???, ???] += 1  # int(t), int(p) GT=t, Pred=p 카운트 증가
    per_class_acc = cm.diagonal().float() / torch.clamp(cm.sum(dim=1).float(), min=1.0)  # 클래스별 정확도
    return cm, per_class_acc

images_all, preds_all, labels_all = collect_predictions(model, test_loader)  # test 전체 예측 수집
cm, pacc = confusion_and_perclass(preds_all, labels_all, num_classes)  # confusion + per-class

# Vision Transformer

In [4]:
from einops import rearrange, repeat  # ViT에서 자주 쓰는 텐서 재배열(rearrange)·복제(repeat)
from einops.layers.torch import Rearrange
from torch import nn, einsum

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)  # 채널별 평균(R,G,B)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)  # 채널별 표준편차(R,G,B)

train_tfms = T.Compose([  # train transform(augmentation 포함)
    T.RandomCrop(32, padding=4),      # 32x32를 padding 후 random crop
    T.RandomHorizontalFlip(),         # 좌우 반전
    T.ToTensor(),                     # PIL -> torch tensor (C,H,W)
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),  # 정규화
])

test_tfms = T.Compose([  # test transform(augmentation 없음)
    T.ToTensor(),                     # 텐서 변환
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),  # 정규화
])

data_root = './data'  # 데이터 저장 경로
train_set = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tfms)  # train set
test_set  = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tfms)  # test set

def denormalize_cifar10(img_chw: torch.Tensor):
    """정규화된 CIFAR-10 텐서(C,H,W)를 시각화용(0~1)으로 되돌립니다."""
    mean = torch.tensor(CIFAR10_MEAN, device=img_chw.device)[:, None, None]
    std  = torch.tensor(CIFAR10_STD,  device=img_chw.device)[:, None, None]
    x = img_chw * std + mean
    return x.clamp(0, 1)

def pair(t):  # image_size/patch_size를 (H,W) 형태로 통일하는 유틸
    return t if isinstance(t, tuple) else (t, t)

class PreNorm(nn.Module):  # Transformer의 Pre-LN 구조: LayerNorm 후 블록 실행
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.??(dim)  # LayerNorm 토큰 임베딩 차원(dim) 기준 LayerNorm -> 패치별 정규화
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)  # 정규화된 토큰을 Attention/FFN에 전달

class FeedForward(nn.Module):  # Transformer의 FFN(MLP) 블록 -> 패치 자체의 정보를 더 깊게 분석
    def __init__(self, dim, hidden_dim, dropout = 0.):
        super().__init__()
        self.net = nn.Sequential(  # FFN: Linear→GELU→Dropout→Linear→Dropout 구성
            nn.???(dim, hidden_dim),
            nn.???(),
            nn.Dropout(dropout),
            nn.???(hidden_dim, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):  # 멀티헤드 Self-Attention(MHSA) 구현
    def __init__(self, dim, heads = 4, dim_head = 64, dropout = 0.):
        super().__init__()
        inner_dim = ??? *  ???  # 전체 헤드 차원 = head 수 × head 차원
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads  # 멀티헤드 개수 저장
        self.scale = dim_head ** -0.5  # Scaled dot-product를 위한 스케일(1/√d)

        self.attend = nn.Softmax(dim = -1)  # attention score를 확률로 변환(softmax), 마지막 차원 : sequence length N
        self.last_attn = None  # (학습/추론 시) 마지막 forward에서의 attention map 저장용(시각화/분석)  # attention score를 확률로 변환(softmax), 마지막 차원 : sequence length N
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias = False)  # 입력 토큰→Q,K,V를 한 번에 선형 변환

        self.to_out = nn.Sequential(  # 헤드들을 합친 뒤 출력 투영 + 드롭아웃
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads   # x = [batch Size, tokens, embedding dimension]
        qkv = self.to_qkv(x).chunk(3, dim = -1)  # 선형변환 결과를 Q,K,V로 분할 -> (Q, K ,V)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h = h), qkv)  # Very Important!!! multi-head 연산을 위해 각 head 기준 n x d 로 분리
        
        dots = einsum('b h i d, b h j d -> b h i j', q, k) * self.scale  # Very Important!!! 각 토큰 간 유사도(Q·Kᵀ) 계산 후 스케일 적용

        attn = self.attend(dots)  # 토큰 간 가중치(attention map) 생성
        self.last_attn = attn.detach()  # 그래프에서 분리(detach)해서 저장 (시각화 목적)

        out = einsum('b h i j, b h j d -> b h i d', attn, v)  # attention 가중합으로 새로운 토큰 표현(out) 계산
        out = rearrange(out, 'b h n d -> b n (h d)')  # 헤드 차원을 다시 합쳐 (batch, tokens, dim)로 복원
        return self.to_out(out)  # 최종 투영을 거쳐 Attention 블록 출력 반환

class Transformer(nn.Module):  # Encoder block을 여러 층(depth) 쌓는 Transformer
    def __init__(self, dim, depth, heads, dim_head, mlp_dim, dropout = 0.):
        super().__init__()
        self.layers = nn.ModuleList([])  # 각 층(Attention+FFN)을 담을 컨테이너
        for _ in range(depth):  # depth 만큼 Encoder block 반복 생성
            self.layers.append(nn.ModuleList([  # 한 층 = (PreNorm+Attention) + (PreNorm+FFN)
                ???(dim, ???(dim, heads = heads, dim_head = dim_head, dropout = dropout)), 
                ???(dim, ???(dim, mlp_dim, dropout = dropout))
            ]))

    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x  # Residual 연결: Attention 결과를 입력에 더해 정보 보존
            x = ff(x) + x  # Residual 연결: FFN 결과를 입력에 더해 정보 보존
        return x

In [ ]:
from dataclasses import dataclass  # 설정값을 묶어 init 인자를 단순화하기 위한 도구
from typing import Tuple, Union  # 파이썬 버전 호환을 위한 타입 힌트

@dataclass
class ViTConfig:
    # 입력/토큰화 관련
    image_size: 'Union[int, Tuple[int, int]]'          # 입력 이미지 크기 (H, W) 또는 정수(정수면 정사각형)
    patch_size: 'Union[int, Tuple[int, int]]'          # 패치 크기 (Ph, Pw) 또는 정수
    channels: int = 3                          # 입력 채널 수 (CIFAR-10=3, MNIST=1)

    # 모델 본체(Encoder) 관련
    dim: int = 64                              # 토큰 임베딩 차원(Transformer의 hidden size)
    depth: int = 6                             # Encoder 블록 개수(= Transformer layer 수)
    heads: int = 4                             # Multi-Head Self-Attention의 head 개수
    dim_head: int = 64                         # 각 head의 Q/K/V 차원
    mlp_dim: int = 128                         # FFN(MLP) 중간 차원

    # 분류/출력 관련
    num_classes: int = 10                      # 분류 클래스 수
    pool: str = "cls"                          # "cls": CLS 토큰 사용, "mean": 토큰 평균 풀링
    dropout: float = 0.0                       # Encoder 내부 dropout(Attention/FFN)
    emb_dropout: float = 0.0                   # 패치+포지션 임베딩 단계 dropout


class ViT(nn.Module):
    """Vision Transformer (ViT)
    - 이미지를 패치로 쪼개 토큰 시퀀스를 만들고
    - CLS 토큰 + 위치 임베딩을 더한 뒤
    - Transformer Encoder로 전역(Self-Attention) 관계를 학습하여
    - CLS(또는 mean pool) 표현으로 분류합니다.
    """

    def __init__(self, cfg: ViTConfig):
        super().__init__()

        # 1) 입력 해상도/패치 크기를 (H,W) 튜플로 정규화
        image_height, image_width = pair(cfg.image_size)
        patch_height, patch_width = pair(cfg.patch_size)

        # 2) 패치가 이미지에 딱 나누어 떨어져야 (h, w) 그리드가 정확히 형성됨
        assert image_height % patch_height == 0 and image_width % patch_width == 0, "Image dimensions must be divisible by the patch size."

        # 3) 패치 개수(=토큰 개수)와 한 패치의 펼친 차원 계산
        num_patches = (image_height // patch_height) * (image_width // patch_width)
        patch_dim = cfg.channels * patch_height * patch_width

        # 4) 풀링 방식 검증: CLS 토큰을 쓸지 mean pool을 쓸지 선택
        assert cfg.pool in {"cls", "mean"}, "pool must be 'cls' or 'mean'"

        # 5) 패치 토큰화: (B,C,H,W) -> (B, N, patch_dim) -> (B, N, dim) 
        self.to_patch_embedding = nn.Sequential(
            # 패치 그리드로 자른 뒤, 각 패치를 1D 벡터로 펼쳐 토큰 시퀀스를 만듦
            Rearrange(
                "b c (h p1) (w p2) -> b (h w) (p1 p2 c)", # Very Important!!!
                p1=patch_height,
                p2=patch_width,
            ),
            # 펼친 패치 벡터를 Transformer hidden size(dim)로 선형 투영
            nn.Linear(patch_dim, cfg.dim), # Very Important!!!
        )

        # 6) CLS 토큰 + 위치 임베딩(학습 파라미터)
        self.cls_token = nn.Parameter(torch.randn(1, 1, cfg.dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, cfg.dim))
        self.dropout = nn.Dropout(cfg.emb_dropout)

        # 7) Transformer Encoder 스택(Attention + FFN + Residual/PreNorm)
        self.transformer = Transformer(
            dim=cfg.dim,
            depth=cfg.depth,
            heads=cfg.heads,
            dim_head=cfg.dim_head,
            mlp_dim=cfg.mlp_dim,
            dropout=cfg.dropout,
        )

        self.pool = cfg.pool

        # 8) 분류 헤드: (CLS/mean) 표현 -> LayerNorm -> Linear(logits)
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(cfg.dim),
            nn.Linear(cfg.dim, cfg.num_classes),
        )

    def forward(self, img):
        # A) 패치 임베딩으로 토큰 시퀀스 생성: (B,C,H,W) -> (B,N,dim)
        x = self.to_patch_embedding(img)
        b, n, _ = x.shape

        # B) CLS 토큰을 배치만큼 복제해 시퀀스 맨 앞에 붙임: (B,1,dim)
        cls_tokens = repeat(self.cls_token, "??? -> ???", b=b) ## 1 1 d -> b 1 d
        x = torch.cat(???, ???)  # (cls_tokens, x), dim=1  (B, N+1, dim) 

        # C) 위치 임베딩을 더해 토큰 순서(공간 위치) 정보를 주입
        x = x + self.pos_embedding[:, : (???)]   # (B, N+1, dim) + (1, N+1, dim) -> Broadcasting
        x = self.dropout(x)

        # D) Encoder를 통과하며 전역 의존성(Self-Attention) 학습
        x = self.transformer(x)  # (B, N+1, dim)

        # E) 이미지 표현 벡터 선택: CLS 토큰(0번) 또는 mean pooling
        x = x[:, 0] if self.pool == "cls" else x.mean(dim=1)   # (B, dim)

        # F) 최종 logits 출력 (softmax는 CrossEntropyLoss 내부에서 처리)
        return self.mlp_head(x)

In [ ]:
def train_epoch(model, optimizer, data_loader, loss_history):
    """한 epoch 학습을 수행하고, 진행 상황을 epoch당 '딱 5번'만 출력합니다."""
    total_samples = len(data_loader.dataset)
    model.train()

    for i, (data, target) in enumerate(data_loader):
        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = F.log_softmax(model(data), dim=1)  # Softmax. Why does it use dim=1? Qustion.
        loss = F.nll_loss(output, target)           # Negative Log Likelihood
        # nn.CrossEntropyLoss(model(data), target)
        loss.backward()
        optimizer.step()

def evaluate(model, data_loader, loss_history):
    """test_loader로 1회 평가하고 (loss, acc)를 반환합니다."""
    model.eval()

    total_samples = len(data_loader.dataset)
    correct_samples = 0
    total_loss = 0.0

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            output = F.log_softmax(model(data), dim=1)
            loss = F.nll_loss(output, target, reduction='sum') 
            _, pred = torch.max(output, dim=1)
            total_loss += float(loss.item())
            correct_samples += int(pred.eq(target).sum().item())

    avg_loss = total_loss / total_samples
    acc = correct_samples / total_samples

    loss_history.append(avg_loss)
    return avg_loss, acc

for epoch in range(start_epoch + 1, target_epoch + 1):

    # 학습
    train_epoch(model, optimizer, train_loader, train_loss_history)

    # 매 epoch마다 test set으로 1회 평가(accuracy 출력)
    test_loss, test_acc = evaluate(model, test_loader, test_loss_history)

# DEtection TRansformer(DETR)

In [12]:
# COCO classes
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]

valid_classes = [c for c in CLASSES if c != 'N/A']

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

In [ ]:
# [DETR] 입력 전처리 + 박스 좌표 후처리 함수
# - DETR은 COCO로 학습된 ResNet backbone을 사용하므로, ImageNet mean/std 정규화를 맞춰야 합니다.
# - 모델 출력 박스는 (cx, cy, w, h) 형식이며 0~1로 정규화되어 있으므로, (xmin, ymin, xmax, ymax)로 변환 후 원본 이미지 크기로 rescale 합니다.

# standard PyTorch mean-std input image normalization
transform = T.Compose([  # 전처리 파이프라인 정의(논문/코드: data augmentation과 정규화)
    T.???(800),  # Resize, 짧은 변을 800에 맞추는 리사이즈(공식 데모 설정)
    T.ToTensor(),  # PIL → torch.Tensor (C,H,W), 0~1 스케일
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet mean/std로 정규화(backbone 사전학습 분포와 정합)
])

# for output bounding box post-processing
def box_cxcywh_to_xyxy(x):  # (cx,cy,w,h) → (xmin,ymin,xmax,ymax) 변환
    x_c, y_c, w, h = x.unbind(1)  # 각 성분을 분리: x.shape=(N,4) 가정
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),  # 중심좌표에서 좌상/우하 꼭짓점 계산. Very important!!
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.???(b, dim=???)  # (N,4)로 다시 스택

def rescale_bboxes(out_bbox, size):  # 정규화 박스를 픽셀 좌표로 변환
    img_w, img_h = size  # PIL size = (W,H)
    b = box_cxcywh_to_xyxy(out_bbox)  # 우선 xyxy로 변환
    b = b * torch.tensor([???, ???, ???, ???], dtype=torch.float32)  # (W,H,W,H) 스케일로 곱해 픽셀 단위로 확장
    return b


In [ ]:
# [DETR] 한 장 이미지 추론 + 신뢰도 필터링 + 박스 스케일 복원
# - outputs['pred_logits']: (1, num_queries, num_classes+1) 로짓. 마지막 '+1'은 'no-object' 클래스.
# - outputs['pred_boxes']: (1, num_queries, 4) 박스 (cx,cy,w,h) 정규화 좌표.
# num_queries: 한 장의 이미지에서 탐색 가능한 최대 객체 개수

model = torch.hub.load('facebookresearch/detr', 'detr_resnet50', pretrained=True)  # 가중치 포함 pretrained DETR-ResNet50 로드
model.eval();  # Dropout/BN 등을 추론 모드로 전환
img = transform(im).unsqueeze(0)  # 전처리 후 배치 차원 추가: (1,3,H,W)
outputs = model(img)  # DETR forward → 로짓/박스 예측

# keep only predictions with 0.9+ confidence
probas = outputs['pred_logits'].softmax(???)[0, :, :???]  # 쿼리별 클래스 확률 계산(no-object 제외): shape=(num_queries,num_classes)
keep = probas.max(-1).values > 0.9  # 각 쿼리의 max class score가 임계값을 넘는 것만 선택

bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, ???], im.size)  # 선택된 박스를 픽셀 좌표로 변환

# U-Net

In [ ]:
nn.ConvTranspose2d(

In [ ]:
# =========================
# 3) U-Net 모델 정의
# =========================
# 이 섹션의 목표:
# - U-Net의 "인코더(Down) → 보틀넥 → 디코더(Up) + Skip Connection" 구조를
#   PyTorch 코드로 직접 따라가며 이해합니다.
#
# 핵심 아이디어(한 줄):
# - Down에서 공간 해상도(H,W)는 줄이고 채널(C)은 늘리면서 특징을 추출하고,
#   Up에서 해상도를 복원하면서 Down 단계의 특징맵을 Skip으로 concat하여
#   localization(위치 정보)을 되살립니다.

class DoubleConv(nn.Module):
    """U-Net 기본 블록: (Conv → ReLU) × 2

    - 첫 번째 Conv가 채널을 '중간 채널(mid_channels)'로 바꾸고,
      두 번째 Conv가 '출력 채널(out_channels)'로 맞춥니다.
    - 기본값(mid_channels=None)일 때는 mid_channels=out_channels로 두어,
      (C_in → C_out → C_out) 형태가 됩니다.

    입력/출력 텐서 형태:
      - 입력:  (B, C_in, H, W)
      - 출력:  (B, C_out, H, W)  # padding=1 이라 H,W 유지
    """
    def __init__(self, in_channels: int, out_channels: int, mid_channels: int | None = None):
        super().__init__()
        if mid_channels is None:
            mid_channels = out_channels

        self.net = nn.Sequential(
            # 1) (C_in → C_mid)
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),

            # 2) (C_mid → C_out)
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class Down(nn.Module):
    """Downsampling 블록: MaxPool(2)로 해상도 1/2 → DoubleConv

    입력/출력 텐서 형태:
      - 입력:  (B, C_in,  H,  W)
      - 출력:  (B, C_out, H/2, W/2)
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.pool = nn.???(kernel_size=2) # MaxPool2d
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(x)           # (B, C_in, H/2, W/2)
        x = self.conv(x)           # (B, C_out, H/2, W/2)
        return x


class Up(nn.Module):
    """Upsampling 블록: 업샘플링 → (Skip concat) → DoubleConv

    구현 관점(중요):
    - in_channels 는 concat 이후 채널 수를 의미합니다.
      예) x1(디코더) 채널=512, x2(skip) 채널=512 → concat 채널=1024 → in_channels=1024

    - bilinear=True:
        1) 업샘플은 파라미터 없는 bilinear interpolation으로 수행
        2) concat 후 DoubleConv에서 mid_channels를 in_channels//2 로 두어
           채널을 자연스럽게 '절반'으로 줄이는 방식(원 논문/레퍼런스 구현과 동일 계열)

    - bilinear=False:
        ConvTranspose2d로 업샘플 자체를 학습(파라미터 증가)
    """
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = False):
        super().__init__()

        if bilinear:
            # (B, C, H/2, W/2) → (B, C, H, W)
            self.up = nn.???(scale_factor=2, mode="bilinear", align_corners=True) # Upsample
            self.conv = DoubleConv(in_channels, out_channels, mid_channels=in_channels // 2) 
        else:
            # (B, C, H/2, W/2) → (B, C/2, H, W)  (deconv가 채널도 절반으로 줄여줌)
            self.up = nn.???(in_channels, in_channels // 2, kernel_size=2, stride=2) # ConvTranspose2d
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        # x1: (B, C_dec, H/2, W/2)  / x2: (B, C_skip, H, W)

        # 1) 업샘플링: 해상도를 skip과 맞추기
        x1 = self.up(x1)  # bilinear: 채널 유지 / deconv: 채널이 절반으로 감소

        # 2) (필요 시) 패딩으로 크기 정렬
        #    - 홀수 크기 입력 등으로 인해 x1과 x2의 H/W가 1~2 픽셀 정도 다를 수 있습니다.
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2,  # left, right
                        diff_y // 2, diff_y - diff_y // 2]) # top, bottom

        # 3) 채널 방향 concat (skip 연결)
        #    cat dim=1 은 채널(C) 축
        x = torch.cat([x2, x1], dim=???)  # (B, C_skip + C_up, H, W) == (B, in_channels, H, W)

        # 4) conv 블록으로 특징 정제 + 채널 축소
        return self.conv(x)

class OutConv(nn.Module):
    """마지막 1x1 conv: 채널을 클래스 수로 매핑

    예)
      - binary segmentation: n_classes=1 (logits 1채널)
      - multi-class:         n_classes=K (logits K채널)
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class UNet(nn.Module):
    """U-Net 전체 모델

    전형적인 채널 구성 예:
      1 → 64 → 128 → 256 → 512 → 1024 (down)
      1024 → 512 → 256 → 128 → 64 (up)
    """
    def __init__(self, n_channels: int, n_classes: int, bilinear: bool = False):
        super().__init__()

        # -------------------------
        # Encoder (Contracting path)
        # -------------------------
        self.inc = DoubleConv(???, 64)   # (B, n_channels, H, W) → (B, 64, H, W)
        self.down1 = Down(64, 128)              # → (B, 128, H/2, W/2)
        self.down2 = Down(128, 256)             # → (B, 256, H/4, W/4)
        self.down3 = Down(256, 512)             # → (B, 512, H/8, W/8)

        # bilinear 업샘플링이면 파라미터/연산을 줄이기 위해 bottleneck 채널을 1024 대신 512로 줄이는 경우가 많음
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)  # → (B, 1024/f, H/16, W/16)

        # -------------------------
        # Decoder (Expanding path)
        # -------------------------
        # Up 블록의 in_channels는 "concat 후 채널" 기준으로 설계되어야 합니다.
        self.up1 = Up(1024, 512 // factor, bilinear)  # (skip=512)와 concat을 고려
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)

        # -------------------------
        # Output head
        # -------------------------
        self.outc = OutConv(64, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # -------------------------
        # Encoder: skip을 위해 각 단계 출력 저장
        # -------------------------
        x1 = self.inc(x)       # (B, 64, H, W)
        x2 = self.down1(x1)    # (B, 128, H/2, W/2)
        x3 = self.down2(x2)    # (B, 256, H/4, W/4)
        x4 = self.down3(x3)    # (B, 512, H/8, W/8)
        x5 = self.down4(x4)    # (B, 1024/f, H/16, W/16)

        # -------------------------
        # Decoder: 업샘플 + skip concat
        # -------------------------
        x = self.up1(???, ???)   # (B, 512/f, H/8, W/8)
        x = self.up2(???, ???)    # (B, 256/f, H/4, W/4)
        x = self.up3(???, ???)    # (B, 128/f, H/2, W/2)
        x = self.up4(???, ???)    # (B, 64, H, W)

        # -------------------------
        # 최종 logits 출력
        # -------------------------
        logits = self.outc(x)  # (B, n_classes, H, W)
        return logits


# =========================
# =========================
# 4) 모델 생성 및 구조 확인
# =========================
model = UNet(n_channels=1, n_classes=1).to(device)

# -------------------------
# 4) 모델 구조 확인: print(model) + torchinfo.summary
# -------------------------
print(model)

# torchinfo.summary는 forward를 한 번 실행해 각 레이어의 출력 shape/파라미터 수를 정리해줍니다.
# - 모델이 학습 모드(train)일 때 BatchNorm/Dropout 등이 있을 수 있으니,
#   구조 확인 목적에서는 eval()로 두는 것이 안전합니다.
model.eval()

# -------------------------
# 5) Forward sanity check (shape 확인)
# -------------------------
dummy = torch.randn(1, 1, 128, 128, device=device)
with torch.no_grad():
    out = model(dummy)

In [ ]:
# 손실 함수 및 옵티마이저 설정
criterion = nn.BCEWithLogitsLoss()  # Sigmoid + Binary Cross Entropy
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# (옵션) 학습 스케줄러
# 20 에포크(Epoch)마다 학습률을 변경, 기존 학습률에 0.5를 곱합
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss


@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    for imgs, masks in dataloader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)
        loss = criterion(logits, masks)
        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss


num_epochs = 10
train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch}/{num_epochs}]  train_loss: {train_loss:.4f}  val_loss: {val_loss:.4f}")